<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_2_slot_runaway_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · 2 — The Stator Slot and Its Thermal Runaway (elliptic)

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L7.2 · Fundamental PDEs**

The copper in the slot of an electrical machine heats up with its current, and
hot copper resists more, so it heats up more. Up to a point that feedback
settles; past it, it would not. How hot is the slot at each current, and where
is that point? You solve it by finite differences and by a physics-informed
network, and score both against the exact solution.

This notebook stands on its own: the die and the panel are the other two
notebooks of the set, and Ex_07.1, which uses the same slot, is not needed.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics - why Poisson becomes Helmholtz | — |
| **4** | the exact solution, and finite differences on a mesh for an agreed 0.1 K | — |
| **5** | a PINN for every current at once, and the comparison | the heat, the residual |
| **6** | what the comparison says | — |
| **7 – 8** | your report, and the mini project | your answers |

About four minutes on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'slot_problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "slot_problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "slot_problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import slot_problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex07.2_outputs"                          # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

A slot in the stator of an electrical machine, its walls held at the cooled
iron's temperature, holding a winding that carries a current. How hot does the
winding get, at currents from 10 to 100 A - three times the rating - and at
what current would the temperature have no steady answer at all?

![One stator slot, with half of each neighbour](https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/Ex07.2_slot.png)

## 2 · The data

| | |
|---|---|
| slot | 10 × 20 mm, walls at 90 °C: a rise of $\theta$ = 0 |
| winding | 32 copper wires of 2 mm, 4 columns of 8, the same current in each; rated 31.42 A |
| bundle | effective conductivity $k_{\text{eff}}$ = 0.70 W/m·K; copper fills 50 % of the slot |
| copper | resistivity 1.72e-08 Ω·m at 20 °C, rising 0.39 % per kelvin |
| currents | 10 to 100 A |

## 3 · The physics

In steady state the heat the winding makes leaves through the walls:

$$k_{\text{eff}}\,\nabla^2\theta + q(\theta, I) = 0, \qquad \theta = 0 \text{ on the walls}$$

The equation is **elliptic**: no time, every wall felt at once. The heat is
$q = f\,\rho(T)\,J^2$ - the copper fill, the resistivity at the local
temperature, the current density - and because $\rho$ rises with $T$, it splits
into a fixed part and a part that grows with the rise, $q = c_0 + c_1\theta$.
Then

$$\nabla^2\theta + \frac{c_1}{k_{\text{eff}}}\,\theta = -\frac{c_0}{k_{\text{eff}}}$$

is a **Helmholtz** equation (L7.2) with $\lambda = c_1/k_{\text{eff}}$, which grows as $I^2$.
The slot's lowest eigenvalue is $\lambda_1 = \pi^2(1/W^2 + 1/H^2)$ = 0.12 per mm². The
feedback multiplies the temperature by about $1/(1 - \lambda/\lambda_1)$, and at
$\lambda = \lambda_1$ there is no steady answer: that is **thermal runaway**,
at 158.4 A for this slot. Long before it the insulation would fail - this is
the model's limit, not an operating point.

**TODO 1** is the heat source, $c_0$ and $c_1$ from the resistivity.

In [ ]:
def heat(I):
    # c0 + c1 theta: the heat at the wall temperature, and how fast it grows with the rise (W/m^3, W/m^3 K)
    J = I / pb.A_WIRE                                           # current density in the copper, A/m^2
    c0 = pb.FILL * pb.RHO_CU * J ** 2 * (1 + pb.ALPHA_CU * (pb.T_WALL - 20.0))
    c1 = pb.FILL * pb.RHO_CU * J ** 2 * pb.ALPHA_CU
    return c0, c1

In [ ]:
W, H = 2 * pb.A_HALF, 2 * pb.B_HALF
lam1 = np.pi ** 2 * (1 / W ** 2 + 1 / H ** 2)                   # the slot's lowest eigenvalue, 1/m^2
c0r, c1r = heat(pb.I_RATED)
I_STAR = pb.I_RATED * np.sqrt(pb.K_EFF * lam1 / c1r)            # where c1 / k_eff reaches lambda_1
check("c0 at rated current, as a ratio", c0r / pb.heat_coefficients(pb.I_RATED)[0], 1.0, tol=1e-9)
print(f"lambda_1 = {lam1 / 1e6:.2f} per mm^2;  thermal runaway at {I_STAR:.2f} A")
for I in (10.0, pb.I_RATED, 60.0, 100.0):
    r = (I / I_STAR) ** 2
    print(f"  {I:6.2f} A: lambda / lambda_1 = {r:.2f}, the feedback multiplies the rise by about {1 / (1 - r):.2f}")

**What you should see.** A `PASS`, $\lambda_1$ = 0.12 per mm², runaway at
158.4 A, and a multiplier of 1.00 at 10 A, 1.04 at rated current, 1.17 at 60 A
and 1.66 at 100 A.

---

## 4 · The reference, and finite differences

**The reference** is exact. The equation has constant coefficients on a
rectangle, so the rise is a sum of the slot's modes, each divided by
$k_{\text{eff}}\lambda_{mn} - c_1$ (`pb.exact_solution`) - and that denominator
is where runaway comes from: at $c_1 = k_{\text{eff}}\lambda_1$ the first term
has none.

**Finite differences** on the five-point stencil, with $c_1\theta$ on the
diagonal: one sparse solve per current (`pb.fdm_solve`). The cell scores four
meshes at 20, 60 and 100 A; **the agreed accuracy is 0.1 K** at every current.

In [ ]:
TARGET = 0.1                                                  # K: the agreed accuracy
TEST_I = np.array([20.0, 40.0, 60.0, 80.0, 100.0])
for nx in (11, 21, 41, 81):
    worst = 0.0
    for I in (20.0, 60.0, 100.0):
        Xg, Yg, th = pb.fdm_solve(I, nx)
        worst = max(worst, np.abs(th - pb.exact_solution(I, Xg[0], Yg[:, 0])).max())
    sec = middle(lambda: pb.fdm_solve(60.0, nx))
    print(f"  {nx:3d} x {2*nx-1} nodes: worst error {worst:.2e} K, one current {sec*1e3:.1f} ms"
          + ("   <- meets 0.1 K" if worst <= TARGET else ""))
NX = 41                                                       # the coarsest mesh that meets it

**What you should see.** The error falling four times with every halving -
6.56e-01, 1.63e-01, 4.08e-02, 1.02e-02 K at worst - and 41 × 81 the first
under 0.1 K, a few milliseconds per current.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(\xi, \eta, s)$ - the position divided by the half-width
and half-depth, and the current mapped to $-1 \ldots 1$ - so **one training answers
every current**. Four layers of 32, 2000 collocation points over the slot and
the current range together.

**The walls are built in** with the mask $(1 - \xi^2)(1 - \eta^2)$, and the output
is scaled by the size of the answer at that current, $20\,(I/I_r)^2/(1 - (I/I^*)^2)$
K - the rated rise, the $I^2$ of the heat, and the feedback's multiplier from
section 3 - so the network learns only the shape.

**TODO 2** is the residual: the slot's equation with the network in place of
$\theta$, divided by $c_0$ so that it is of order one at every current.

In [ ]:
a, b = pb.A_HALF, pb.B_HALF
I_LO, I_HI = 10.0, 100.0
def current(s):
    return I_LO + (s + 1) / 2 * (I_HI - I_LO)                 # s in -1..1 back to amperes

class Slot(torch.nn.Module):                                  # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=32, n_layers=4)       # dense tanh network: xi, eta, s in
    def forward(self, p):
        xi, eta, s = p[:, :1], p[:, 1:2], p[:, 2:]
        I = current(s)
        size = 20.0 * (I / pb.I_RATED) ** 2 / (1 - (I / I_STAR) ** 2)    # K: the size of the answer
        return size * self.net(p) * (1 - xi ** 2) * (1 - eta ** 2)       # the mask: zero on every wall

In [ ]:
def residual(model, p):
    th = model(p)                                             # the network's rise, K
    c0, c1 = heat(current(p[:, 2:]))                          # the heat at each point's current
    lap = d2(th, p, 0) / a ** 2 + d2(th, p, 1) / b ** 2       # the Laplacian by autograd, in metres
    return (pb.K_EFF * lap + c0 + c1 * th) / c0               # the equation, scaled

In [ ]:
pts = to_tensor(interior_points(2000, ((-1, 1), (-1, 1), (-1, 1)), method="lhs", seed=1), requires_grad=True)  # collocation points, differentiable
model = Slot().to(DEVICE)                                     # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, lambda: residual(model, pts).pow(2).mean(),
                adam_steps=1500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
Xg, Yg, _ = pb.fdm_solve(60.0, NX); xs, ys = Xg[0], Yg[:, 0]
Pn = np.c_[Xg.ravel() / a, Yg.ravel() / b]                   # the mesh's nodes, for the network too
def pinn_at(I):
    s = 2 * (I - I_LO) / (I_HI - I_LO) - 1
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(to_tensor(np.c_[Pn, np.full(len(Pn), s)]))).reshape(Xg.shape)   # the network at every node
rows_out, pinn_hot = [], []
for I in TEST_I:
    ex = pb.exact_solution(I, xs, ys)
    u = pinn_at(I)
    rows_out.append((I, ex.max(), np.abs(pb.fdm_solve(I, NX)[2] - ex).max(), np.abs(u - ex).max()))
    pinn_hot.append(u.max())
t_fdm = middle(lambda: pb.fdm_solve(60.0, NX))
t_pin = middle(lambda: pinn_at(60.0))
print(f"{'current':>9s}{'exact hot spot':>16s}{'FDM error':>12s}{'PINN error':>12s}")
for I, hs, ef, ep in rows_out:
    print(f"{I:7.0f} A{hs:14.2f} K{ef:10.2e} K{ep:10.2e} K")
print(f"per current: finite differences {t_fdm*1e3:.1f} ms, PINN {t_pin*1e3:.1f} ms; training {train_time:.0f} s")
fig, ax = plt.subplots(figsize=(6.4, 3.8))
Is = np.linspace(10, 100, 46)
ax.plot(Is, [pb.exact_solution(I, [0.0], [0.0])[0, 0] / (I / pb.I_RATED) ** 2 for I in Is], label="exact")
ax.plot(TEST_I, np.array(pinn_hot) / (TEST_I / pb.I_RATED) ** 2, "o", label="PINN")
ax.axhline(pb.exact_solution(pb.I_RATED, [0.0], [0.0])[0, 0], color="k", lw=0.8, ls="--", label="without the feedback")
ax.set_xlabel("current  [A]"); ax.set_ylabel("hot spot / (I / I_rated)^2  [K]"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex07.2_2_slot_runaway_report.png"), dpi=150); plt.show()

**What you should see.** The exact hot spot rising from 7.40 K at 20 A to
315.86 K at 100 A - far faster than $I^2$, as the curve shows. Finite
differences within 4.08e-02 K at every current, about 8 ms each. The network
within 0.1 K up to 80 A - 2.74e-03 K at 20 A, 5.03e-02 K at 80 A - but about
0.28 K at 100 A; about 2 ms per current, after about three minutes of training.

---

## 6 · What the comparison says

* **The feedback is the physics that matters.** At rated current it adds 4 %; at
  100 A it multiplies the rise by 1.66, and at 158.4 A the slot would have no
  steady temperature. The eigenvalue of section 3 said so before any solver ran.
* **Finite differences meet 0.1 K at every current; the network misses it at
  100 A.** Its error grows with the current and is worst where the feedback is
  strongest - the equation there is nearest to having no answer, and hardest to
  fit.
* **Once trained, the network is faster per current** - about four times - and
  one training answers every current. It pays back after some tens of thousands
  of currents: a design sweep, not one question.

---

## 7 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex07.2_2_slot_runaway_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L7.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Why does the copper's resistance turn the slot's Poisson equation into a Helmholtz equation, and what is lambda? (-> L7.2 Q5)": """
""",
    "What is the slot's first eigenvalue, what happens to the temperature as lambda approaches it, and at what current? (-> L7.2 Q5)": """
""",
    "Why is the network's error largest at the highest current, and what did finite differences need there? (-> L7.2 Q2)": """
""",
    "To conclude: finite differences met 0.1 K at every current. For which slot question would you train a network instead? (-> L7.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| current | exact hot spot | FDM error | PINN error |", "|---|---|---|---|",
            *[f"| {I:.0f} A | {hs:.2f} K | {ef:.2e} K | {ep:.2e} K |" for I, hs, ef, ep in rows_out],
            "", f"Per current: finite differences {t_fdm*1e3:.1f} ms, PINN {t_pin*1e3:.1f} ms; training {train_time:.0f} s."]
    out = ["# Ex_07.2 — The Stator Slot and Its Thermal Runaway", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex07.2_2_slot_runaway_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex07.2_2_slot_runaway_report.md into Ex07.2_2_slot_runaway_report.pdf, with any figure
# saved as Ex07.2_2_slot_runaway_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex07.2_2_slot_runaway_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex07.2_2_slot_runaway_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex07.2_2_slot_runaway_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex07.2_2_slot_runaway_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 8 · Mini project proposal

One problem grown out of this notebook, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP7.2B · The eigenvalues of a real slot

**The problem.** This notebook found the first eigenvalue of a
rectangular slot. A real slot is not a rectangle: the teeth between slots have
parallel sides, so the slot between them widens towards the back of the
stator. Here the slot tapers, and the first three eigenvalues and their modes
are wanted - the first sets the current at which the copper's feedback would
run away.

| | |
|---|---|
| slot | tapered: 8 mm wide at the opening, 12 mm at the bottom, 20 mm deep - the same 200 mm² as the 10 × 20 mm rectangle |
| walls | $u = 0$ on all four |
| equation | $\nabla^2 u + \lambda u = 0$, the first three $\lambda$ and their modes |
| winding | as in the notebook: $k_{\text{eff}} = 0.70$ W/m·K, rated 31.42 A |

**Why a plain PINN is not enough.** Every eigenvalue problem has the trivial answer $u = 0$, and above the first
mode the network finds the first one again unless it is told the new mode
must be orthogonal to it. The walls are no longer the edges of a rectangle,
so the notebook's mask has to be rebuilt for the taper.

**What you build.** An eigenvalue PINN: the mask built from the slot's four walls, a norm term
against the trivial solution, a trainable $\lambda$, and each higher mode
trained orthogonal to the ones found before it. From $\lambda_1$, the runaway
current.

**The ground truth you get.** `MP7.2C_truth.npz`, from the same script: five-point finite differences on
the nodes inside the slot, 0.03125 mm apart (204,417 nodes), the first three
eigenpairs, and the eigenvalues extrapolated from three grids.

**Worked example.** 

| grid | nodes | $\lambda_1$ | $\lambda_2$ | $\lambda_3$ (per mm²) | runaway |
|---|---|---|---|---|---|
| 0.125 mm | 12,705 | 0.1209 | 0.1998 | 0.3232 | 156.8 A |
| 0.0625 mm | 51,009 | 0.1217 | 0.2008 | 0.3243 | 157.3 A |
| 0.03125 mm | 204,417 | 0.1222 | 0.2013 | 0.3248 | 157.6 A |
| extrapolated | | 0.1226 | 0.2018 | 0.3253 | 157.9 A |

The walls are drawn in steps on the grid, so the eigenvalues converge at first
order, and the extrapolation uses the order the three grids show (0.99 to
1.03). The rectangle's $\lambda_1$ is 0.1234 per mm² and its runaway 158.4 A:
tapering the slot lowers $\lambda_1$ by only **0.7 %**.

**Requirements.**

* $\lambda_1$ within **0.3 %** - enough to say whether the tapered slot runs away before the rectangle.
* $\lambda_2$ and $\lambda_3$ within **1 %**, each mode correlating with the ground truth's at **0.99** or better.
* The runaway current within **0.5 A**.

**What you hand in.** The three eigenvalues and modes against the ground truth, the runaway
current, and what orthogonality cost your training. Say which of the two slot
shapes a designer should prefer, and whether the difference matters.